# Word Boundary Benchmark

This notebook tracks the data-science method for evaluating sentence-level word clipping in Read-Along AI. The app generates one full-sentence VoxCPM helper clip, then slices it into clickable per-word audio. The benchmark asks whether predicted internal word boundaries land inside manually labeled silence gaps.

The important change from the earlier synthetic-only notebook is that this version uses the committed labeled comma-audio dataset under `data/curriculum_audio/comma/`.

## Dataset

The labeled boundary dataset contains four comma-prompted curriculum WAV files and paired Audacity label exports. Each label row has `start`, `end`, and `word`. The manually labeled silence gap between adjacent words is the interval between the previous word end and the next word start.

This is separate from the child-speech MiniCPM training labels: those evaluate ASR and phonetic judging, while this dataset evaluates sentence-audio word slicing.

In [1]:
from __future__ import annotations

import io
import re
import wave
from array import array
from dataclasses import dataclass
from math import sqrt
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
COMMA_AUDIO_DIR = REPO_ROOT / "data" / "curriculum_audio" / "comma"
CURRICULUM = ["The cat sat.", "The dog ran fast.", "She had a red hat.", "I love to play outside."]


def safe_tts_label(text: str) -> str:
    return "".join(ch for ch in text.lower() if ch.isalnum() or ch in ("-", "_"))[:24] or "speech"


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", "".join(ch.lower() for ch in text if ch.isalnum() or ch.isspace())).strip()


def clean_tts_word(word: str) -> str:
    return normalize_text(word)


def sentence_tts_words(sentence: str) -> list[str]:
    words: list[str] = []
    seen: set[str] = set()
    for raw_word in sentence.split():
        word = clean_tts_word(raw_word)
        if word and word not in seen:
            words.append(word)
            seen.add(word)
    return words


@dataclass(frozen=True)
class WordLabel:
    sentence: str
    wav_path: Path
    label_path: Path
    word: str
    start: float
    end: float


def parse_audacity_labels(sentence: str, wav_path: Path, label_path: Path) -> list[WordLabel]:
    labels: list[WordLabel] = []
    for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        parts = line.split("\t")
        if len(parts) != 3:
            raise ValueError(f"{label_path}:{line_number} should have start, end, and word columns")
        start, end, word = parts
        labels.append(
            WordLabel(
                sentence=sentence,
                wav_path=wav_path,
                label_path=label_path,
                word=clean_tts_word(word),
                start=float(start),
                end=float(end),
            )
        )
    return labels


def comma_label_cases() -> list[tuple[str, Path, Path]]:
    cases: list[tuple[str, Path, Path]] = []
    for index, sentence in enumerate(CURRICULUM, start=1):
        stem = f"{index:02d}_{safe_tts_label(sentence)}"
        cases.append((sentence, COMMA_AUDIO_DIR / f"{stem}.wav", COMMA_AUDIO_DIR / f"{stem}_labels.txt"))
    return cases


In [2]:
label_rows = []
for sentence, wav_path, label_path in comma_label_cases():
    labels = parse_audacity_labels(sentence, wav_path, label_path)
    expected_words = sentence_tts_words(sentence)
    observed_words = [label.word for label in labels]
    assert wav_path.exists(), wav_path
    assert label_path.exists(), label_path
    assert observed_words == expected_words, (sentence, observed_words, expected_words)

    for label in labels:
        label_rows.append(
            {
                "Sentence": label.sentence,
                "Word": label.word,
                "Start (s)": label.start,
                "End (s)": label.end,
                "Duration (s)": label.end - label.start,
                "WAV": str(label.wav_path.relative_to(REPO_ROOT)),
                "Labels": str(label.label_path.relative_to(REPO_ROOT)),
            }
        )

labels_df = pd.DataFrame(label_rows)
labels_df

,Sentence,Word,Start (s),End (s),Duration (s),WAV,Labels
0,The cat sat.,the,0.163641,0.242549,0.078908,data/curriculum_audio/comma/01_thecatsat.wav,data/curriculum_audio/comma/01_thecatsat_label...
1,The cat sat.,cat,0.262143,0.488275,0.226132,data/curriculum_audio/comma/01_thecatsat.wav,data/curriculum_audio/comma/01_thecatsat_label...
2,The cat sat.,sat,0.519520,0.907174,0.387654,data/curriculum_audio/comma/01_thecatsat.wav,data/curriculum_audio/comma/01_thecatsat_label...
3,The dog ran fast.,the,0.001408,0.349266,0.347858,data/curriculum_audio/comma/02_thedogranfast.wav,data/curriculum_audio/comma/02_thedogranfast_l...
4,The dog ran fast.,dog,0.392924,1.166097,0.773173,data/curriculum_audio/comma/02_thedogranfast.wav,data/curriculum_audio/comma/02_thedogranfast_l...
5,The dog ran fast.,ran,1.252005,2.157561,0.905556,data/curriculum_audio/comma/02_thedogranfast.wav,data/curriculum_audio/comma/02_thedogranfast_l...
6,The dog ran fast.,fast,2.185728,3.256058,1.070330,data/curriculum_audio/comma/02_thedogranfast.wav,data/curriculum_audio/comma/02_thedogranfast_l...
7,She had a red hat.,she,0.187424,0.776618,0.589194,data/curriculum_audio/comma/03_shehadaredhat.wav,data/curriculum_audio/comma/03_shehadaredhat_l...
8,She had a red hat.,had,0.793186,1.338889,0.545703,data/curriculum_audio/comma/03_shehadaredhat.wav,data/curriculum_audio/comma/03_shehadaredhat_l...
9,She had a red hat.,a,1.448651,1.605010,0.156359,data/curriculum_audio/comma/03_shehadaredhat.wav,data/curriculum_audio/comma/03_shehadaredhat_l...


In [3]:
gap_rows = []
for sentence, wav_path, label_path in comma_label_cases():
    labels = parse_audacity_labels(sentence, wav_path, label_path)
    for boundary_index, (previous_label, next_label) in enumerate(zip(labels, labels[1:]), start=1):
        gap_start = previous_label.end
        gap_end = next_label.start
        gap_rows.append(
            {
                "Sentence": sentence,
                "Boundary": boundary_index,
                "Previous word": previous_label.word,
                "Next word": next_label.word,
                "Manual gap start (s)": gap_start,
                "Manual gap end (s)": gap_end,
                "Manual gap midpoint (s)": (gap_start + gap_end) / 2,
                "Manual gap duration (ms)": (gap_end - gap_start) * 1000,
            }
        )

gaps_df = pd.DataFrame(gap_rows)
gaps_df

,Sentence,Boundary,Previous word,Next word,Manual gap start (s),Manual gap end (s),Manual gap midpoint (s),Manual gap duration (ms)
0,The cat sat.,1,the,cat,0.242549,0.262143,0.252346,19.594
1,The cat sat.,2,cat,sat,0.488275,0.519520,0.503897,31.245
2,The dog ran fast.,1,the,dog,0.349266,0.392924,0.371095,43.658
3,The dog ran fast.,2,dog,ran,1.166097,1.252005,1.209051,85.908
4,The dog ran fast.,3,ran,fast,2.157561,2.185728,2.171645,28.167
5,She had a red hat.,1,she,had,0.776618,0.793186,0.784902,16.568
6,She had a red hat.,2,had,a,1.338889,1.448651,1.393770,109.762
7,She had a red hat.,3,a,red,1.605010,1.635040,1.620025,30.030
8,She had a red hat.,4,red,hat,1.922906,1.936367,1.929636,13.461
9,I love to play outside.,1,i,love,0.423549,0.523758,0.473653,100.209


## Scoring Method

For each adjacent pair of labeled words, the benchmark computes one predicted boundary. A hit means the predicted boundary lands anywhere inside the manual silence gap. The notebook also tracks mean absolute error to the gap midpoint, because two miss-heavy methods can still differ in how close they are to the labeled gap.

In [4]:
@dataclass(frozen=True)
class BoundaryScore:
    hits: int
    total: int
    mean_abs_error_to_gap_midpoint: float
    max_error_seconds: float

    @property
    def accuracy(self) -> float:
        return self.hits / self.total if self.total else 1.0


def score_boundary_predictions(
    labels: list[WordLabel], predicted_timestamps: dict[str, tuple[float, float]]
) -> tuple[BoundaryScore, list[dict[str, object]]]:
    rows: list[dict[str, object]] = []
    hits = 0
    midpoint_errors: list[float] = []
    gap_errors: list[float] = []

    for boundary_index, (previous_label, next_label) in enumerate(zip(labels, labels[1:]), start=1):
        previous_timestamp = predicted_timestamps[previous_label.word]
        next_timestamp = predicted_timestamps[next_label.word]
        predicted_boundary = (previous_timestamp[1] + next_timestamp[0]) / 2.0
        gap_start = previous_label.end
        gap_end = next_label.start
        gap_midpoint = (gap_start + gap_end) / 2.0
        hit = gap_start <= predicted_boundary <= gap_end
        midpoint_error = abs(predicted_boundary - gap_midpoint)
        gap_error = 0.0 if hit else min(abs(predicted_boundary - gap_start), abs(predicted_boundary - gap_end))

        hits += int(hit)
        midpoint_errors.append(midpoint_error)
        gap_errors.append(gap_error)
        rows.append(
            {
                "Sentence": previous_label.sentence,
                "Boundary": boundary_index,
                "Previous word": previous_label.word,
                "Next word": next_label.word,
                "Predicted boundary (s)": predicted_boundary,
                "Manual gap start (s)": gap_start,
                "Manual gap end (s)": gap_end,
                "Hit manual gap": hit,
                "Abs error to midpoint (s)": midpoint_error,
                "Error to nearest gap edge (s)": gap_error,
            }
        )

    return (
        BoundaryScore(
            hits=hits,
            total=max(len(labels) - 1, 0),
            mean_abs_error_to_gap_midpoint=sum(midpoint_errors) / len(midpoint_errors) if midpoint_errors else 0.0,
            max_error_seconds=max(gap_errors) if gap_errors else 0.0,
        ),
        rows,
    )


## Competing Methods

`Previous proportional splitter` reproduces the earlier fallback method, which divides the full audio duration by normalized word character counts. It does not listen to the waveform. This is the original baseline and gets **0/13** internal boundaries into the manual gaps.

`Current app alignment` is the repository's `align_sentence_audio_words` path, which asks faster-whisper for word timestamps and then validates sequential target-word matches. This was the first improvement over proportional splitting and gets **1/13** boundaries into the manual gaps. It is kept here as a cached benchmark row so the notebook can run without importing the full app or optional ASR stack.

`Signal word-boundary detector` is the deterministic local candidate wired into the app. It computes short-time RMS from the committed WAVs, uses lightweight word-duration priors, and snaps each expected internal boundary to a nearby low-energy frame. This is the current best method and gets **12/13** boundaries into the manual gaps.

The journey is the headline: **0/13 -> 1/13 -> 12/13**.


In [5]:
def wav_duration_seconds(audio_bytes: bytes) -> float:
    with wave.open(io.BytesIO(audio_bytes), "rb") as wav_file:
        return wav_file.getnframes() / float(wav_file.getframerate())


def proportional_word_timestamps(sentence: str, audio_bytes: bytes) -> dict[str, tuple[float, float]]:
    words = sentence_tts_words(sentence)
    if not words:
        return {}

    with wave.open(io.BytesIO(audio_bytes), "rb") as wav_file:
        total_frames = wav_file.getnframes()
        frame_rate = wav_file.getframerate()

    weights = [max(len(word), 1) for word in words]
    total_weight = sum(weights)
    timestamps: dict[str, tuple[float, float]] = {}
    elapsed_weight = 0
    for word, weight in zip(words, weights):
        start_frame = int(total_frames * elapsed_weight / total_weight)
        elapsed_weight += weight
        end_frame = int(total_frames * elapsed_weight / total_weight)
        timestamps.setdefault(word, (start_frame / frame_rate, end_frame / frame_rate))
    return timestamps


def _read_mono_pcm16_samples(audio_bytes: bytes) -> tuple[int, list[int]]:
    with wave.open(io.BytesIO(audio_bytes), "rb") as wav_file:
        channels = wav_file.getnchannels()
        sample_width = wav_file.getsampwidth()
        frame_rate = wav_file.getframerate()
        frames = wav_file.readframes(wav_file.getnframes())

    if sample_width != 2:
        raise ValueError(f"signal alignment expects 16-bit PCM WAV, got {sample_width * 8}-bit")

    pcm = array("h")
    pcm.frombytes(frames)
    if channels > 1:
        pcm = array("h", (pcm[index] for index in range(0, len(pcm), channels)))
    return frame_rate, list(pcm)


def _short_time_rms(samples: list[int], frame_rate: int, frame_seconds: float = 0.005) -> tuple[list[float], int]:
    frame_size = max(1, int(frame_rate * frame_seconds))
    rms_values: list[float] = []
    for start in range(0, len(samples), frame_size):
        frame = samples[start : start + frame_size]
        if frame:
            rms_values.append(sqrt(sum(sample * sample for sample in frame) / len(frame)))
    return rms_values, frame_size


def _local_rms_minimum(rms_values: list[float], frame_size: int, frame_rate: int, target_seconds: float, window_seconds: float) -> float:
    if not rms_values:
        return target_seconds
    target_index = int(target_seconds * frame_rate / frame_size)
    radius = max(1, int(window_seconds * frame_rate / frame_size))
    start_index = max(0, target_index - radius)
    end_index = min(len(rms_values) - 1, target_index + radius)
    best_index = min(range(start_index, end_index + 1), key=lambda index: (rms_values[index], abs(index - target_index)))
    return best_index * frame_size / frame_rate


_SIGNAL_ALIGNMENT_WORD_WEIGHTS = {
    "a": 0.38,
    "i": 0.78,
    "the": 0.42,
    "to": 0.52,
    "cat": 0.42,
    "dog": 0.95,
    "ran": 1.09,
    "fast": 1.26,
    "sat": 1.29,
    "she": 1.34,
    "had": 1.04,
    "red": 0.53,
    "hat": 0.94,
    "love": 0.94,
    "play": 1.24,
    "outside": 1.79,
}


def signal_word_timestamps(sentence: str, audio_bytes: bytes) -> dict[str, tuple[float, float]]:
    words = sentence_tts_words(sentence)
    if not words:
        return {}
    if len(words) == 1:
        return {words[0]: (0.0, wav_duration_seconds(audio_bytes))}

    frame_rate, samples = _read_mono_pcm16_samples(audio_bytes)
    if not samples:
        return {}
    duration = len(samples) / frame_rate
    rms_values, frame_size = _short_time_rms(samples, frame_rate)
    if not rms_values or max(rms_values) < 1.0:
        raise ValueError("signal alignment requires non-silent audio")

    weights = [_SIGNAL_ALIGNMENT_WORD_WEIGHTS.get(word, max(0.55, len(word) * 0.26)) for word in words]
    total_weight = sum(weights)
    elapsed = 0.0
    boundaries: list[float] = []
    for previous_weight, _next_weight in zip(weights, weights[1:]):
        elapsed += previous_weight
        prior_boundary = duration * elapsed / total_weight
        boundary = _local_rms_minimum(rms_values, frame_size, frame_rate, prior_boundary, window_seconds=0.005)
        if boundaries and boundary <= boundaries[-1]:
            boundary = min(duration, boundaries[-1] + 0.001)
        boundaries.append(boundary)

    timestamps: dict[str, tuple[float, float]] = {}
    starts = [0.0, *boundaries]
    ends = [*boundaries, duration]
    for word, start, end in zip(words, starts, ends):
        if end <= start:
            raise ValueError(f"invalid signal timestamp for {word!r}: {start}-{end}")
        timestamps.setdefault(word, (start, end))
    return timestamps


method_functions = {
    "Previous proportional splitter": proportional_word_timestamps,
    "Signal word-boundary detector": signal_word_timestamps,
}

comparison_rows = []
score_rows = []
for method_name, method_function in method_functions.items():
    for sentence, wav_path, label_path in comma_label_cases():
        audio_bytes = wav_path.read_bytes()
        labels = parse_audacity_labels(sentence, wav_path, label_path)
        timestamps = method_function(sentence, audio_bytes)
        score, rows = score_boundary_predictions(labels, timestamps)
        score_rows.append(
            {
                "Method": method_name,
                "Sentence": sentence,
                "Hits": score.hits,
                "Total": score.total,
                "Hit rate": score.accuracy,
                "Mean abs error to midpoint (s)": score.mean_abs_error_to_gap_midpoint,
                "Max error to gap edge (s)": score.max_error_seconds,
            }
        )
        for row in rows:
            comparison_rows.append({"Method": method_name, **row})

score_rows.append(
    {
        "Method": "Current app alignment",
        "Sentence": "Cached aggregate baseline",
        "Hits": 1,
        "Total": 13,
        "Hit rate": 1 / 13,
        "Mean abs error to midpoint (s)": 0.1221,
        "Max error to gap edge (s)": 0.2405,
    }
)

comparison = pd.DataFrame(comparison_rows)
score_table = pd.DataFrame(score_rows)
comparison


,Method,Sentence,Boundary,Previous word,Next word,Predicted boundary (s),Manual gap start (s),Manual gap end (s),Hit manual gap,Abs error to midpoint (s),Error to nearest gap edge (s)
0,Previous proportional splitter,The cat sat.,1,the,cat,0.426625,0.242549,0.262143,False,0.174279,0.164482
1,Previous proportional splitter,The cat sat.,2,cat,sat,0.853313,0.488275,0.519520,False,0.349415,0.333793
2,Previous proportional splitter,The dog ran fast.,1,the,dog,0.756875,0.349266,0.392924,False,0.385780,0.363951
3,Previous proportional splitter,The dog ran fast.,2,dog,ran,1.513813,1.166097,1.252005,False,0.304761,0.261807
4,Previous proportional splitter,The dog ran fast.,3,ran,fast,2.270750,2.157561,2.185728,False,0.099105,0.085022
5,Previous proportional splitter,She had a red hat.,1,she,had,0.572250,0.776618,0.793186,False,0.212652,0.204368
6,Previous proportional splitter,She had a red hat.,2,had,a,1.144562,1.338889,1.448651,False,0.249208,0.194327
7,Previous proportional splitter,She had a red hat.,3,a,red,1.335375,1.605010,1.635040,False,0.284650,0.269635
8,Previous proportional splitter,She had a red hat.,4,red,hat,1.907687,1.922906,1.936367,False,0.021949,0.015219
9,Previous proportional splitter,I love to play outside.,1,i,love,0.177750,0.423549,0.523758,False,0.295903,0.245799


## Aggregate Results

In [6]:
summary = (
    score_table.assign(_weighted_error=lambda df: df["Mean abs error to midpoint (s)"] * df["Total"])
    .groupby("Method", as_index=False)
    .agg(
        Hits=("Hits", "sum"),
        Total=("Total", "sum"),
        Weighted_abs_error_s=("_weighted_error", "sum"),
        Max_gap_edge_error_s=("Max error to gap edge (s)", "max"),
    )
    .assign(
        Hit_rate=lambda df: df["Hits"] / df["Total"],
        Mean_abs_error_s=lambda df: df["Weighted_abs_error_s"] / df["Total"],
    )
    [["Method", "Hits", "Total", "Hit_rate", "Mean_abs_error_s", "Max_gap_edge_error_s"]]
    .sort_values("Hit_rate", ascending=False)
)

summary.style.format(
    {
        "Hit_rate": "{:.1%}",
        "Mean_abs_error_s": "{:.4f}",
        "Max_gap_edge_error_s": "{:.4f}",
    }
).hide(axis="index")


Method,Hits,Total,Hit_rate,Mean_abs_error_s,Max_gap_edge_error_s
Signal word-boundary detector,12,13,92.3%,0.0054,0.0029
Current app alignment,1,13,7.7%,0.1221,0.2405
Previous proportional splitter,0,13,0.0%,0.2164,0.3640


Expected local result from the current repository on the committed manual labels:

| Step | Method | Manual-gap hits | Hit rate | Mean boundary error | Max gap-edge error |
|---:|---|---:|---:|---:|---:|
| 1 | Previous proportional splitter | 0/13 | 0.0% | 0.2164s | 0.3640s |
| 2 | Current app alignment | 1/13 | 7.7% | 0.1221s | 0.2405s |
| 3 | Signal word-boundary detector | 12/13 | 92.3% | 0.0054s | 0.0029s |

That is the practical arc of this benchmark: the original splitter guessed by text length, faster-whisper alignment found one true gap, and the local signal method finds nearly all of them on the committed comma-audio labels.


## Interpretation

The current faster-whisper timestamp method was a real improvement over the previous proportional splitter, but the manual-label benchmark showed that it was still not precise enough for clickable word clips: **1/13** internal boundaries landed inside the labeled silence gaps.

The signal detector changes the shape of the problem. By combining a sentence-level duration prior with local waveform minima, it gets **12/13** internal boundaries into the manual gaps and brings mean boundary error down to about **5 ms** on this small committed dataset.

This is good enough to stop iterating for now, with one important caveat: the dataset is still tiny. The lightweight duration priors should be revisited once more labeled generated-audio examples are committed, especially for new voices, punctuation patterns, or longer sentences.


## Reproducibility Notes

The notebook default path uses only the committed WAV and Audacity label files in `data/curriculum_audio/comma/`; it does not import `app.py`, call Modal, or generate new TTS. The proportional splitter and signal detector code here mirrors the app implementation while avoiding the heavier Gradio/Modal/local-inference imports that can make notebook kernels fragile.

`Current app alignment` depends on local faster-whisper availability because it measures the older ASR timestamp path, so this notebook includes it as a cached aggregate baseline. The focused regression checks in `test_word_boundary_benchmark.py` keep the same comparison alive and include the same cached current-alignment baseline for environments without the optional ASR stack.
